In [1]:
import sys
sys.path.insert(0, '/home/jy/tza-pypsa')

# Now your imports will use the local version
import pypsa
from tz_pypsa.constraints import (
    constr_max_annual_utilisation_generator, 
    constr_min_annual_utilisation_generator,
    constr_max_annual_utilisation_links,
    constr_min_annual_utilisation_links,
    constr_max_annual_utilisation_storage_discharge, 
    constr_min_annual_utilisation_storage_discharge, 
    constr_max_annual_utilisation_storage_charge,     
    constr_min_annual_utilisation_storage_charge,     
    constr_soc_intraday_profile,
    constr_soc_weekly_profile,
    constr_production_target_max,
    constr_production_target_min,
    constr_max_ramps_daily
)

import plotly.express as px
import pandas as pd     
import numpy as np
import xarray as xr
import os
os.environ['GRB_LICENSE_FILE'] = '/home/jy/opt/gurobi/gurobi.lic'

In [2]:
n = pypsa.Network()
# n.import_from_netcdf("/home/jy/Backup/client-earth_CE-A-OCCTO-003_v3/platform_network.nc") # calibration
n.import_from_netcdf("/home/jy/tz-amp/.amp/client-earth_CE-A-SEP7-005_v1/platform_network.nc")

INFO:pypsa.io:Imported network platform_network.nc has buses, carriers, generators, links, loads, storage_units


In [3]:
n.generators_t.marginal_cost.loc[:, n.generators_t.marginal_cost.filter(like="coal").columns] = n.generators_t.marginal_cost.loc[:, n.generators_t.marginal_cost.filter(like="coal-unspecified").columns].max().max()
n.generators_t.marginal_cost.loc[:, n.generators_t.marginal_cost.filter(like="gas").columns] = n.generators_t.marginal_cost.loc[:, n.generators_t.marginal_cost.filter(like="gas-unspecified").columns].max().max()

In [4]:
n_solved = pypsa.Network()
n_solved.import_from_netcdf("/home/jy/tz-amp/.amp/client-earth_CE-A-SEP7-005_da-d4/platform_network.solved.nc")

INFO:pypsa.io:Imported network platform_network.solved.nc has buses, carriers, generators, links, loads, storage_units


In [5]:
n.generators.loc[n_solved.generators.p_nom_extendable, "p_nom"] = np.ceil(n_solved.generators.loc[n_solved.generators.p_nom_extendable, "p_nom_opt"])
n.storage_units.loc[n_solved.storage_units.p_nom_extendable, "p_nom"] = np.ceil(n_solved.storage_units.loc[n_solved.storage_units.p_nom_extendable, "p_nom_opt"])

In [6]:
new_nuclear_df = pd.read_csv("nuclear_asset_level_data.csv", index_col=0)

In [7]:
old_nuclear_names = n.generators[n.generators.type == "nuclear"].index
n.mremove("Generator", old_nuclear_names)

In [8]:
n.import_components_from_dataframe(new_nuclear_df, "Generator")

In [ ]:
n_nuc = pypsa.Network()
n_nuc.import_from_netcdf("/home/jy/tz-amp/.amp/client-earth_CE-A-SEP7-005_v1/platform_network.nc")

In [ ]:
n.generators.loc[n.generators.type == 'nuclear', 'p_nom'] = n_nuc.generators.loc[n_nuc.generators.type == 'nuclear', 'p_nom']

In [ ]:
n.generators.loc[:, "p_nom"] = np.ceil(n_solved.generators.loc[:, "p_nom_opt"])
n.storage_units.loc[:, "p_nom"] = np.ceil(n_solved.storage_units.loc[:, "p_nom_opt"])

In [9]:
n.generators['carrier'] = n.generators['type']
n.links['carrier'] = n.links['type']
n.storage_units['carrier'] = n.storage_units['type']

In [10]:
all_carriers = (
    n.generators.carrier.unique().tolist()
    + n.storage_units.carrier.unique().tolist()
    + n.links.carrier.unique().tolist()
)
missing_carriers = set(all_carriers) - set(n.carriers.index)
if missing_carriers:
    n.add("Carrier", missing_carriers)

n.generators.build_year = 2040
n.storage_units.build_year = 2040
n.links.build_year = 2040

In [ ]:
n.storage_units_t.state_of_charge_set[:] = float("nan")

In [ ]:
n.storage_units_t.state_of_charge_set[n.storage_units_t.state_of_charge_set.notna().any(axis=1)]

In [ ]:
n.generators.loc[n.generators.carrier == 'wind-offshore-unspecified', 'p_nom_extendable'] = True
n.generators.loc[n.generators.carrier == 'wind-onshore', 'p_nom_extendable'] = True
n.generators.loc[n.generators.carrier == 'photovoltaic-unspecified', 'p_nom_extendable'] = True
n.generators.loc[n.generators.carrier == 'geothermal-unspecified', 'p_nom_extendable'] = True

n.generators.loc[n.generators.carrier == 'gas-ccs', 'p_nom_extendable'] = True
n.generators.loc[n.generators.carrier == 'coal-ammonia-cofiring', 'p_nom_extendable'] = True
n.generators.loc[n.generators.carrier == 'gas-hydrogen-cofiring', 'p_nom_extendable'] = True
n.storage_units.loc[n.storage_units.carrier == 'utility-scale', 'p_nom_extendable'] = True

In [ ]:
# p_nom for nuclear across different scenario
# Scenario A - 20% nuclear generation share which is the default capacity configuration on DWH

# Scenario B - 12% nuclear generation share
# n.generators.loc[(n.generators.carrier == 'nuclear') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'p_nom'] = 2070
# n.generators.loc[(n.generators.carrier == 'nuclear') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'p_nom'] = 825
# n.generators.loc[(n.generators.carrier == 'nuclear') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'p_nom'] = 2712
# n.generators.loc[(n.generators.carrier == 'nuclear') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'p_nom'] = 0
# n.generators.loc[(n.generators.carrier == 'nuclear') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'p_nom'] = 1206
# n.generators.loc[(n.generators.carrier == 'nuclear') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'p_nom'] = 4100
# n.generators.loc[(n.generators.carrier == 'nuclear') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'p_nom'] = 820
# n.generators.loc[(n.generators.carrier == 'nuclear') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'p_nom'] = 890
# n.generators.loc[(n.generators.carrier == 'nuclear') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'p_nom'] = 4140

# Scenario C - 16% nuclear generation share
n.generators.loc[(n.generators.carrier == 'nuclear') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'p_nom'] = 2070
n.generators.loc[(n.generators.carrier == 'nuclear') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'p_nom'] = 2208
n.generators.loc[(n.generators.carrier == 'nuclear') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'p_nom'] = 3812
n.generators.loc[(n.generators.carrier == 'nuclear') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'p_nom'] = 0
n.generators.loc[(n.generators.carrier == 'nuclear') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'p_nom'] = 1206
n.generators.loc[(n.generators.carrier == 'nuclear') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'p_nom'] = 6578
n.generators.loc[(n.generators.carrier == 'nuclear') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'p_nom'] = 2193
n.generators.loc[(n.generators.carrier == 'nuclear') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'p_nom'] = 890
n.generators.loc[(n.generators.carrier == 'nuclear') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'p_nom'] = 4140

In [ ]:
# set p_nom_min = p_nom to avoid capacity retirement
n.generators.p_nom_min = n.generators.p_nom

In [ ]:
# Renewable p_nom_max
# p_nom_max for geothermal
n.generators.loc[n.generators.carrier == 'geothermal-unspecified', 'p_nom_max'] = n.generators.loc[n.generators.carrier == 'geothermal-unspecified', 'p_nom'] * 1.5

# p_nom_max for solar
n.generators.loc[(n.generators.carrier == 'photovoltaic-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'p_nom_max'] = 8305 
n.generators.loc[(n.generators.carrier == 'photovoltaic-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'p_nom_max'] = 33780 
n.generators.loc[(n.generators.carrier == 'photovoltaic-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'p_nom_max'] = 60231 
n.generators.loc[(n.generators.carrier == 'photovoltaic-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'p_nom_max'] = 38999 
n.generators.loc[(n.generators.carrier == 'photovoltaic-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'p_nom_max'] = 4970 
n.generators.loc[(n.generators.carrier == 'photovoltaic-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'p_nom_max'] = 23097 
n.generators.loc[(n.generators.carrier == 'photovoltaic-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'p_nom_max'] = 26285 
n.generators.loc[(n.generators.carrier == 'photovoltaic-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'p_nom_max'] = 13496 
n.generators.loc[(n.generators.carrier == 'photovoltaic-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'p_nom_max'] = 49404

# p_nom_max for onshore wind
n.generators.loc[(n.generators.carrier == 'wind-onshore') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'p_nom_max'] = 6290
n.generators.loc[(n.generators.carrier == 'wind-onshore') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'p_nom_max'] = 18246
n.generators.loc[(n.generators.carrier == 'wind-onshore') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'p_nom_max'] = 3890
n.generators.loc[(n.generators.carrier == 'wind-onshore') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'p_nom_max'] = 1221
n.generators.loc[(n.generators.carrier == 'wind-onshore') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'p_nom_max'] = 1789
n.generators.loc[(n.generators.carrier == 'wind-onshore') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'p_nom_max'] = 2404
n.generators.loc[(n.generators.carrier == 'wind-onshore') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'p_nom_max'] = 2144
n.generators.loc[(n.generators.carrier == 'wind-onshore') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'p_nom_max'] = 1950
n.generators.loc[(n.generators.carrier == 'wind-onshore') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'p_nom_max'] = 3039

# p_nom_max for offshore wind
# Taking highest quality sites for each region
n.generators.loc[(n.generators.carrier == 'wind-offshore-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'p_nom_max'] = 19375 + n.generators.loc[(n.generators.carrier == 'wind-offshore-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'p_nom_min']
n.generators.loc[(n.generators.carrier == 'wind-offshore-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'p_nom_max'] = 3167 + n.generators.loc[(n.generators.carrier == 'wind-offshore-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'p_nom_min']
n.generators.loc[(n.generators.carrier == 'wind-offshore-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'p_nom_max'] = 7123 + n.generators.loc[(n.generators.carrier == 'wind-offshore-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'p_nom_min']
n.generators.loc[(n.generators.carrier == 'wind-offshore-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'p_nom_max'] = 10378 + n.generators.loc[(n.generators.carrier == 'wind-offshore-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'p_nom_min']
n.generators.loc[(n.generators.carrier == 'wind-offshore-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'p_nom_max'] = 3891 + n.generators.loc[(n.generators.carrier == 'wind-offshore-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'p_nom_min']
n.generators.loc[(n.generators.carrier == 'wind-offshore-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'p_nom_max'] = 1110 + n.generators.loc[(n.generators.carrier == 'wind-offshore-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'p_nom_min']
n.generators.loc[(n.generators.carrier == 'wind-offshore-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'p_nom_max'] = 843 + n.generators.loc[(n.generators.carrier == 'wind-offshore-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'p_nom_min']
n.generators.loc[(n.generators.carrier == 'wind-offshore-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'p_nom_max'] = 1982 + n.generators.loc[(n.generators.carrier == 'wind-offshore-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'p_nom_min']
n.generators.loc[(n.generators.carrier == 'wind-offshore-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'p_nom_max'] = 134 + n.generators.loc[(n.generators.carrier == 'wind-offshore-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'p_nom_min']

In [ ]:
# p_nom_max for coal-ammonia-cofiring
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'p_nom_max'] = 280 + n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'p_nom_min']
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'p_nom_max'] = 1035 + n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'p_nom_min']
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'p_nom_max'] = 1166 + n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'p_nom_min']
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'p_nom_max'] = 661 + n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'p_nom_min']
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'p_nom_max'] = 344 + n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'p_nom_min']
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'p_nom_max'] = 670 + n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'p_nom_min']
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'p_nom_max'] = 824 + n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'p_nom_min']
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'p_nom_max'] = 573 + n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'p_nom_min']
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'p_nom_max'] = 902 + n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'p_nom_min']

In [ ]:
# p_nom_max for gas-hydrogen-cofiring
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'p_nom_max'] = 105 + n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'p_nom_min']
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'p_nom_max'] = 652 + n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'p_nom_min']
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'p_nom_max'] = 2585 + n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'p_nom_min']
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'p_nom_max'] = 1200 + n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'p_nom_min']
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'p_nom_max'] = 74 + n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'p_nom_min']
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'p_nom_max'] = 893 + n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'p_nom_min']
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'p_nom_max'] = 190 + n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'p_nom_min']
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'p_nom_max'] = 85 + n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'p_nom_min']
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'p_nom_max'] = 402 + n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'p_nom_min']

In [ ]:
n.generators.groupby('carrier')[['p_nom', 'p_nom_max']].sum()

In [11]:
n.generators.loc[(n.generators.carrier == 'coal-unspecified'), 'ramp_limit_up'] = 0.4
n.generators.loc[(n.generators.carrier == 'coal-unspecified'), 'ramp_limit_down'] = 0.4

n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring'), 'ramp_limit_up'] = 0.4
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring'), 'ramp_limit_down'] = 0.4

n.generators.loc[(n.generators.carrier == 'gas-unspecified'), 'ramp_limit_up'] = 0.9
n.generators.loc[(n.generators.carrier == 'gas-unspecified'), 'ramp_limit_down'] = 0.9

n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring'), 'ramp_limit_up'] = 0.9
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring'), 'ramp_limit_down'] = 0.9

n.generators.loc[(n.generators.carrier == 'gas-ccs'), 'ramp_limit_up'] = 0.9
n.generators.loc[(n.generators.carrier == 'gas-ccs'), 'ramp_limit_down'] = 0.9

n.generators.loc[(n.generators.carrier == 'nuclear'), 'ramp_limit_up'] = 0.6
n.generators.loc[(n.generators.carrier == 'nuclear'), 'ramp_limit_down'] = 0.6

In [12]:
n.storage_units.loc[n.storage_units.carrier == 'utility-scale', 'efficiency_store'] = 0.92
n.storage_units.loc[n.storage_units.carrier == 'utility-scale', 'efficiency_dispatch'] = 0.92

In [13]:
n.generators.loc[n.generators.carrier == 'nuclear', 'marginal_cost'] = 1
n.generators.loc[n.generators.carrier == 'wind-offshore-unspecified', 'marginal_cost'] = -1
n.generators.loc[n.generators.carrier == 'wind-onshore', 'marginal_cost'] = -1
n.generators.loc[n.generators.carrier == 'photovoltaic-unspecified', 'marginal_cost'] = -1

In [14]:
n.generators.loc[n.generators.carrier == 'nuclear', 'p_min_pu'] = 0.7
n.generators.loc[n.generators.carrier == 'nuclear', 'p_max_pu'] = 0.5

In [ ]:
n.generators_t.marginal_cost.loc[:, n.generators_t.marginal_cost.filter(like="gas-ccs").columns] = 163.88

In [15]:
n.generators.loc[n.generators.carrier == 'nuclear', 'max_ramps_per_day'] = 2

In [16]:
n.generators_t.p_min_pu = n.generators_t.p_max_pu.filter(regex='biomass|geothermal|hydro')

In [17]:
# p_max_pu - coal
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'p_max_pu'] = 0.46
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'p_max_pu'] = 0.57
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'p_max_pu'] = 0.62
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'p_max_pu'] = 0.63
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'p_max_pu'] = 0.60
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'p_max_pu'] = 0.60
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'p_max_pu'] = 0.52
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'p_max_pu'] = 0.59
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'p_max_pu'] = 0.51

# p_max_pu - coal-ammonia cofiring
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'p_max_pu'] = 0.46
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'p_max_pu'] = 0.57
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'p_max_pu'] = 0.62
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'p_max_pu'] = 0.63
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'p_max_pu'] = 0.60
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'p_max_pu'] = 0.60
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'p_max_pu'] = 0.52
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'p_max_pu'] = 0.59
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'p_max_pu'] = 0.51

In [18]:
# max_utilisation_rate
# coal
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'max_utilisation_rate'] = 0.34
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'max_utilisation_rate'] = 0.445
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'max_utilisation_rate'] = 0.52
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'max_utilisation_rate'] = 0.55
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'max_utilisation_rate'] = 0.51
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'max_utilisation_rate'] = 0.52
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'max_utilisation_rate'] = 0.41
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'max_utilisation_rate'] = 0.45
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'max_utilisation_rate'] = 0.365

# coal-ammonia cofiring
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'max_utilisation_rate'] = 0.34
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'max_utilisation_rate'] = 0.445
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'max_utilisation_rate'] = 0.52
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'max_utilisation_rate'] = 0.55
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'max_utilisation_rate'] = 0.51
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'max_utilisation_rate'] = 0.52
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'max_utilisation_rate'] = 0.41
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'max_utilisation_rate'] = 0.45
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'max_utilisation_rate'] = 0.365

# # coal-ammonia cofiring
# n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'max_utilisation_rate'] = 0.70
# n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'max_utilisation_rate'] = 0.70
# n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'max_utilisation_rate'] = 0.70
# n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'max_utilisation_rate'] = 0.70
# n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'max_utilisation_rate'] = 0.70
# n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'max_utilisation_rate'] = 0.70
# n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'max_utilisation_rate'] = 0.70
# n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'max_utilisation_rate'] = 0.70
# n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'max_utilisation_rate'] = 0.70

# # gas - calibration constraints
# n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'max_utilisation_rate'] = 0.095
# n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'max_utilisation_rate'] = 0.16
# n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'max_utilisation_rate'] = 0.23
# n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'max_utilisation_rate'] = 0.10
# n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'max_utilisation_rate'] = 0.225
# n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'max_utilisation_rate'] = 0.05
# n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'max_utilisation_rate'] = 0.09
# n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'max_utilisation_rate'] = 0.04

# gas
n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'max_utilisation_rate'] = 0.70


# gas-ccs
n.generators.loc[(n.generators.carrier == 'gas-ccs') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-ccs') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-ccs') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-ccs') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-ccs') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-ccs') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-ccs') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-ccs') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-ccs') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'max_utilisation_rate'] = 0.70

# gas-hydrogen-cofiring
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'max_utilisation_rate'] = 0.70

In [19]:
# min_utilisation_rate
# coal
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'min_utilisation_rate'] = 0.34
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'min_utilisation_rate'] = 0.445
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'min_utilisation_rate'] = 0.52
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'min_utilisation_rate'] = 0.365

# coal-ammonia-cofiring
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'min_utilisation_rate'] = 0.34
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'min_utilisation_rate'] = 0.445
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'min_utilisation_rate'] = 0.52
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'min_utilisation_rate'] = 0.365

# # Assume coal-ammonia-cofiring to follow the same min utilisation rate as gas to reflect the same level of operational constraints
# n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'min_utilisation_rate'] = 0.14
# n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'min_utilisation_rate'] = 0.095
# n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'min_utilisation_rate'] = 0.16
# n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'min_utilisation_rate'] = 0.23
# n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'min_utilisation_rate'] = 0.10
# n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'min_utilisation_rate'] = 0.225
# n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'min_utilisation_rate'] = 0.05
# n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'min_utilisation_rate'] = 0.085
# n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'min_utilisation_rate'] = 0.04

# # gas
# n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'min_utilisation_rate'] = 0.34
# n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'min_utilisation_rate'] = 0.445
# n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'min_utilisation_rate'] = 0.52
# n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'min_utilisation_rate'] = 0.365

# gas - calibration constraints
n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'min_utilisation_rate'] = 0.14
n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'min_utilisation_rate'] = 0.095
n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'min_utilisation_rate'] = 0.16
n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'min_utilisation_rate'] = 0.23
n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'min_utilisation_rate'] = 0.10
n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'min_utilisation_rate'] = 0.225
n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'min_utilisation_rate'] = 0.05
n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'min_utilisation_rate'] = 0.085
n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'min_utilisation_rate'] = 0.04

# Assume gas-hydrogen-cofiring to follow the same min utilisation rate as gas to reflect the same level of operational constraints
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'min_utilisation_rate'] = 0.14
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'min_utilisation_rate'] = 0.095
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'min_utilisation_rate'] = 0.16
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'min_utilisation_rate'] = 0.23
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'min_utilisation_rate'] = 0.10
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'min_utilisation_rate'] = 0.225
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'min_utilisation_rate'] = 0.05
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'min_utilisation_rate'] = 0.085
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'min_utilisation_rate'] = 0.04

# Assume gas-ccs to follow the same min utilisation rate as gas to reflect the same level of operational constraints
n.generators.loc[(n.generators.carrier == 'gas-ccs') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'min_utilisation_rate'] = 0.14
n.generators.loc[(n.generators.carrier == 'gas-ccs') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'min_utilisation_rate'] = 0.095
n.generators.loc[(n.generators.carrier == 'gas-ccs') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'min_utilisation_rate'] = 0.16
n.generators.loc[(n.generators.carrier == 'gas-ccs') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'min_utilisation_rate'] = 0.23
n.generators.loc[(n.generators.carrier == 'gas-ccs') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'min_utilisation_rate'] = 0.10
n.generators.loc[(n.generators.carrier == 'gas-ccs') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'min_utilisation_rate'] = 0.225
n.generators.loc[(n.generators.carrier == 'gas-ccs') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'min_utilisation_rate'] = 0.05
n.generators.loc[(n.generators.carrier == 'gas-ccs') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'min_utilisation_rate'] = 0.085
n.generators.loc[(n.generators.carrier == 'gas-ccs') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'min_utilisation_rate'] = 0.04

In [20]:
# max_utilisation_rate - transmission
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-HK~GRIDREGION-JPN-TH'), 'max_utilisation_rate'] = 0.48
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-TH~GRIDREGION-JPN-HK'), 'max_utilisation_rate'] = 0.00
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-TH~GRIDREGION-JPN-TK'), 'max_utilisation_rate'] = 0.5875
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-TK~GRIDREGION-JPN-TH'), 'max_utilisation_rate'] = 0.00
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-TK~GRIDREGION-JPN-CB'), 'max_utilisation_rate'] = 0.44
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-CB~GRIDREGION-JPN-SH'), 'max_utilisation_rate'] = 0.00
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-CB~GRIDREGION-JPN-TK'), 'max_utilisation_rate'] = 0.25
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-CB~GRIDREGION-JPN-HR'), 'max_utilisation_rate'] = 0.02
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-CB~GRIDREGION-JPN-KA'), 'max_utilisation_rate'] = 0.04
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-HR~GRIDREGION-JPN-CB'), 'max_utilisation_rate'] = 0.33
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-HR~GRIDREGION-JPN-KA'), 'max_utilisation_rate'] = 0.18
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-KA~GRIDREGION-JPN-HR'), 'max_utilisation_rate'] = 0.09
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-KA~GRIDREGION-JPN-CB'), 'max_utilisation_rate'] = 0.25
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-KA~GRIDREGION-JPN-CG'), 'max_utilisation_rate'] = 0.19
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-KA~GRIDREGION-JPN-SH'), 'max_utilisation_rate'] = 0.00
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-CG~GRIDREGION-JPN-KA'), 'max_utilisation_rate'] = 0.40
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-CG~GRIDREGION-JPN-SH'), 'max_utilisation_rate'] = 0.00
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-SH~GRIDREGION-JPN-KY'), 'max_utilisation_rate'] = 0.00
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-KY~GRIDREGION-JPN-CG'), 'max_utilisation_rate'] = 0.34
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-KY~GRIDREGION-JPN-SH'), 'max_utilisation_rate'] = 0.83

In [21]:
# min_utilisation_rate - transmission
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-HK~GRIDREGION-JPN-TH'), 'min_utilisation_rate'] = 0.48
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-TH~GRIDREGION-JPN-TK'), 'min_utilisation_rate'] = 0.5875
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-TK~GRIDREGION-JPN-CB'), 'min_utilisation_rate'] = 0.44
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-CB~GRIDREGION-JPN-TK'), 'min_utilisation_rate'] = 0.25
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-CB~GRIDREGION-JPN-KA'), 'min_utilisation_rate'] = 0.04
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-HR~GRIDREGION-JPN-CB'), 'min_utilisation_rate'] = 0.32
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-KA~GRIDREGION-JPN-CB'), 'min_utilisation_rate'] = 0.25
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-KA~GRIDREGION-JPN-HR'), 'min_utilisation_rate'] = 0.09
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-KA~GRIDREGION-JPN-CG'), 'min_utilisation_rate'] = 0.19
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-CG~GRIDREGION-JPN-KA'), 'min_utilisation_rate'] = 0.40
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-CG~GRIDREGION-JPN-KY'), 'min_utilisation_rate'] = 0.19
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-SH~GRIDREGION-JPN-KA'), 'min_utilisation_rate'] = 0.98
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-SH~GRIDREGION-JPN-CG'), 'min_utilisation_rate'] = 0.91

In [22]:
# max_utilisation_rate
# hydro-pumped-storage
n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-HK'), 'discharge_min_utilisation_rate'] = 0.126
n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-HK'), 'charge_max_utilisation_rate'] = 0.180

n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-TH'), 'discharge_min_utilisation_rate'] = 0.136
n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-TH'), 'charge_max_utilisation_rate'] = 0.195

n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-TK'), 'discharge_min_utilisation_rate'] = 0.121
n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-TK'), 'charge_max_utilisation_rate'] = 0.174

n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-CB'), 'discharge_min_utilisation_rate'] = 0.059
n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-CB'), 'charge_max_utilisation_rate'] = 0.085

n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-HR'), 'discharge_min_utilisation_rate'] = 0.055
n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-HR'), 'charge_max_utilisation_rate'] = 0.080

n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-KA'), 'discharge_min_utilisation_rate'] = 0.056
n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-KA'), 'charge_max_utilisation_rate'] = 0.080

n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-CG'), 'discharge_min_utilisation_rate'] = 0.056
n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-CG'), 'charge_max_utilisation_rate'] = 0.081

n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-SH'), 'discharge_min_utilisation_rate'] = 0.064
n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-SH'), 'charge_max_utilisation_rate'] = 0.092

n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-KY'), 'discharge_min_utilisation_rate'] = 0.057
n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-KY'), 'charge_max_utilisation_rate'] = 0.083

In [ ]:
n.generators.loc[n.generators.carrier == 'gas-unspecified']

In [23]:
n.optimize.create_model()
# constr_max_annual_utilisation_generator(n, carriers='coal-unspecified|gas-unspecified') # Calibration
# constr_min_annual_utilisation_generator(n, carriers='coal-unspecified|gas-unspecified') # Calibration
constr_max_annual_utilisation_generator(n, carriers='coal-unspecified|gas-unspecified|gas-hydrogen-cofiring|coal-ammonia-cofiring|gas-ccs') # Set max annual utilisation for these generators
constr_min_annual_utilisation_generator(n, carriers='coal-unspecified|gas-unspecified|gas-hydrogen-cofiring|coal-ammonia-cofiring|gas-ccs') # Set min annual utilisation for these generators
constr_max_annual_utilisation_links(n, carriers='transmission') # Set max annual utilisation for these links
constr_min_annual_utilisation_links(n, carriers='transmission') # Set min annual utilisation for these links
constr_min_annual_utilisation_storage_discharge(n, carriers='hydro-pumped-storage-unspecified') # Set min annual utilisation for these storage units
constr_max_annual_utilisation_storage_charge(n, carriers='hydro-pumped-storage-unspecified') # Set min annual utilisation for these storage units
constr_soc_intraday_profile(
    n, 
    max_csv="/home/jy/Backup/client-earth_CE-A-OCCTO-003_testing/templates/state_of_charge_intraday_profile_annual_max.csv",
    min_csv="/home/jy/Backup/client-earth_CE-A-OCCTO-003_testing/templates/state_of_charge_intraday_profile_annual_min.csv"
)
constr_soc_weekly_profile(
    n, 
    max_csv="/home/jy/Backup/client-earth_CE-A-OCCTO-003_testing/templates/state_of_charge_weekly_profile_annual_max.csv",
    min_csv="/home/jy/Backup/client-earth_CE-A-OCCTO-003_testing/templates/state_of_charge_weekly_profile_annual_min.csv",
    day_shift=2
)
# constr_production_target_min(n, 
#                             ['GRIDREGION-JPN-SH', 'GRIDREGION-JPN-HR', 'GRIDREGION-JPN-CB', 
#                             'GRIDREGION-JPN-HK', 'GRIDREGION-JPN-KA', 'GRIDREGION-JPN-CG', 
#                             'GRIDREGION-JPN-TK', 'GRIDREGION-JPN-TH', 'GRIDREGION-JPN-KY'],
#                             ['wind-offshore-unspecified','photovoltaic-unspecified', 'wind-onshore', 'geothermal-unspecified', 'biomass', 'hydro-reservoir-and-run-of-river'],
#                             0.50)

# constr_production_target_max(n, 
#                             ['GRIDREGION-JPN-SH', 'GRIDREGION-JPN-HR', 'GRIDREGION-JPN-CB', 
#                             'GRIDREGION-JPN-HK', 'GRIDREGION-JPN-KA', 'GRIDREGION-JPN-CG', 
#                             'GRIDREGION-JPN-TK', 'GRIDREGION-JPN-TH', 'GRIDREGION-JPN-KY'],
#                             ['wind-offshore-unspecified','photovoltaic-unspecified', 'wind-onshore', 'geothermal-unspecified', 'biomass', 'hydro-reservoir-and-run-of-river'],
#                             0.50)

constr_max_ramps_daily(n, carriers='nuclear')

MultiIndex([(2040, '2040-01-01 00:00:00'),
            (2040, '2040-01-01 01:00:00'),
            (2040, '2040-01-01 02:00:00'),
            (2040, '2040-01-01 03:00:00'),
            (2040, '2040-01-01 04:00:00'),
            (2040, '2040-01-01 05:00:00'),
            (2040, '2040-01-01 06:00:00'),
            (2040, '2040-01-01 07:00:00'),
            (2040, '2040-01-01 08:00:00'),
            (2040, '2040-01-01 09:00:00'),
            ...
            (2040, '2040-12-31 14:00:00'),
            (2040, '2040-12-31 15:00:00'),
            (2040, '2040-12-31 16:00:00'),
            (2040, '2040-12-31 17:00:00'),
            (2040, '2040-12-31 18:00:00'),
            (2040, '2040-12-31 19:00:00'),
            (2040, '2040-12-31 20:00:00'),
            (2040, '2040-12-31 21:00:00'),
            (2040, '2040-12-31 22:00:00'),
            (2040, '2040-12-31 23:00:00')],
           names=['period', 'timestep'], length=8760)
MultiIndex([(2040, '2040-01-01 00:00:00'),
            (2040, '2040-0

['gas-ccs:GRIDREGION-JPN-CB', 'gas-ccs:GRIDREGION-JPN-CG', 'gas-ccs:GRIDREGION-JPN-HK', 'gas-ccs:GRIDREGION-JPN-HR', 'gas-ccs:GRIDREGION-JPN-KA', 'gas-ccs:GRIDREGION-JPN-KY', 'gas-ccs:GRIDREGION-JPN-SH', 'gas-ccs:GRIDREGION-JPN-TH', 'gas-ccs:GRIDREGION-JPN-TK', 'coal-unspecified:GRIDREGION-JPN-CB', 'coal-unspecified:GRIDREGION-JPN-CG', 'coal-unspecified:GRIDREGION-JPN-HK', 'coal-unspecified:GRIDREGION-JPN-HR', 'coal-unspecified:GRIDREGION-JPN-KA', 'coal-unspecified:GRIDREGION-JPN-KY', 'coal-unspecified:GRIDREGION-JPN-SH', 'coal-unspecified:GRIDREGION-JPN-TH', 'coal-unspecified:GRIDREGION-JPN-TK', 'gas-unspecified:GRIDREGION-JPN-CB', 'gas-unspecified:GRIDREGION-JPN-CG', 'gas-unspecified:GRIDREGION-JPN-HK', 'gas-unspecified:GRIDREGION-JPN-HR', 'gas-unspecified:GRIDREGION-JPN-KA', 'gas-unspecified:GRIDREGION-JPN-KY', 'gas-unspecified:GRIDREGION-JPN-SH', 'gas-unspecified:GRIDREGION-JPN-TH', 'gas-unspecified:GRIDREGION-JPN-TK', 'coal-ammonia-cofiring:GRIDREGION-JPN-CB', 'coal-ammonia-cofiri

In [25]:
n.generators[['e_sum_max']]

,e_sum_max
Generator,
wind-offshore-unspecified:GRIDREGION-JPN-CB,inf
wind-offshore-unspecified:GRIDREGION-JPN-CG,inf
wind-offshore-unspecified:GRIDREGION-JPN-HK,inf
wind-offshore-unspecified:GRIDREGION-JPN-HR,inf
wind-offshore-unspecified:GRIDREGION-JPN-KA,inf
...,...
nuclear-Takahama:GRIDREGION-JPN-KA,inf
nuclear-Shimane:GRIDREGION-JPN-CG,inf
nuclear-Ikata:GRIDREGION-JPN-SH,inf


In [ ]:
n.model.constraints['max_ramps_per_day']

In [ ]:
n.optimize.solve_model(
    solver_name='gurobi',
    solver_options={
        'threads': 8,
        'method': 2, # barrier
        'crossover': 0,
        'BarConvTol': 1.e-6,
        'Seed': 123,
        'AggFill': 0,
        'PreDual': 0,
        'LogFile': 'gurobi.log',
        'LogToConsole': 1  
    },
    io_api="direct",
    env=None,
)

In [ ]:
n.generators_t.p.filter(regex='nuclear').sum() / (n.generators.p_nom.filter(regex='nuclear') * 8760)

In [ ]:
n.storage_units_t.state_of_charge.filter(regex='GRIDREGION-JPN-KY')

In [ ]:
n.export_to_netcdf("/home/jy/tz-amp/.amp/client-earth_CE-A-SEP7-005_da-d4-dispatch-testing-vv2-ph/platform_network.solved.nc")

In [ ]:
n.buses_t.marginal_price.describe()

In [ ]:
n.generators_t.p_max_pu

In [ ]:
n.model.print_infeasibilities()

In [ ]:
n.generators.groupby(['carrier'])[['p_nom', 'p_nom_max', 'p_nom_opt']].sum()

In [ ]:
n.buses_t.marginal_price.filter(like='KY')['GRIDREGION-JPN-KY'].round(0).plot.hist(bins=100, logy=True)

In [ ]:
import pandas as pd
pd.Series(n.buses_t.marginal_price.filter(like='KY').values.flatten()).unique()
# Then use matplotlib directly
import matplotlib.pyplot as plt
plt.plot(pd.Series(n.buses_t.marginal_price.filter(like='KY').values.flatten()).unique())
plt.show()

In [ ]:
RE_gen = n.generators_t.p.filter(regex="wind-offshore-unspecified|photovoltaic-unspecified|wind-onshore|geothermal-unspecified|biomass|hydro-reservoir-and-run-of-river").sum().sum()
total_gen = n.generators_t.p.sum().sum()
RE_gen / total_gen

In [ ]:
RE_gen = n.generators_t.p.filter(regex="nuclear").sum().sum()
total_gen = n.generators_t.p.sum().sum()
RE_gen / total_gen

In [ ]:
n.statistics()

In [ ]:
n.generators_t.marginal_cost.filter(regex='gas|coal').mean()

In [ ]:
n.generators.groupby('carrier')[['capital_cost']].sum().astype(int)

In [ ]:
n.generators[['p_nom', 'p_nom_opt', 'carrier']].groupby('carrier').sum()

In [ ]:
n.generators[['p_nom', 'p_nom_opt']]

In [ ]:
import plotly.express as px

px.line(n.generators_t.p.filter(regex='nuclear').reset_index(drop=True, level=0))

In [ ]:
n.generators.loc[n.generators.carrier == 'nuclear']

In [ ]:
n.generators_t.p.filter(regex='nuclear:GRIDREGION-JPN-TK')

In [ ]:
n.storage_units_t.p_dispatch.sum() / (n.storage_units.p_nom_opt * 8760)

In [ ]:
n.storage_units_t.p_dispatch.sum() / (n.storage_units.p_nom_opt * 8760)

In [ ]:
n

In [ ]:
n.storage_units_t.p_dispatch.sum()

In [ ]:
n.storage_units_t.p_dispatch.filter(regex='TK')

In [ ]:
n.storage_units_t.p_store.filter(regex='TK')


In [ ]:
n.storage_units_t.p_dispatch.sum() / n.storage_units_t.p_store.sum()

In [ ]:
import plotly.express as px

px.line(n.storage_units_t.state_of_charge.filter(regex='GRIDREGION-JPN-HK').reset_index(drop=True, level=0))

In [ ]:
n.links_t.p0.sum() / (n.links.p_nom * 8760)

In [ ]:
n.links_t.p0.sum()

In [ ]:
n.storage_units_t.p_dispatch.filter(regex='utility')

In [ ]:
n.model.constraints

In [ ]:
n.storage_units_t.p_store.filter(regex='hydro')

In [ ]:
utility_p = n.storage_units_t.p.filter(regex='utility')
result = utility_p[utility_p > 0]


In [ ]:
result.sum()

In [ ]:
n.statistics(groupby=['carrier'])